# ✍️ 05 — Generation Evaluation

Notebook 04 showed the retriever finds the right chunks. That is necessary, not sufficient: the
answer model can still ignore them, add things they don't say, or answer a different question.
This notebook grades the **answers** in the baseline run, first without a reference answer (the
checks that also work in production), then against the reference answers in the golden set,
and finally splits every failure between the retriever and the generator.

## Learning Objectives
In this notebook, you will learn:
1. **Decline correctness** - did the agent say "I don't know" exactly when it should?
2. **Reference-free metrics** - faithfulness, answer relevancy and hallucination
3. **Reference-based metrics** - exact match, token F1, factual correctness, semantic similarity, a `GEval` judge
4. **Blame** - for each failed answer, was it the retriever, the generator, or the test case?

## Prerequisites
- Notebook 04 run once: it writes `data/golden_runs.json` (the baseline run)
- `OPENAI_API_KEY` in the `.env` file at the repository root

---
## 🗺️ The idea

```
 baseline run record                      golden case
 (question, retrieved chunks, answer)     (reference answer, relevant chunk IDs)
        │                                          │
        ├──► reference-free: is the answer supported by its chunks? does it address the question?
        │
        └──────────────────► reference-based: does the answer say what the reference says?
                                                   │
                     both, plus notebook 04's recall ──► blame: retriever, generator, or test case
```

| Family | Needs | Runs in production? | Metrics here |
|---|---|---|---|
| Reference-free | Question, chunks, answer | Yes | Faithfulness, answer relevancy, hallucination |
| Reference-based | A reference answer | No, only on the golden set | EM, token F1, factual correctness, semantic similarity, `GEval` correctness |

---
## ⚙️ Part 0: Setup

This notebook reads saved runs and never runs the agent, so LangSmith tracing is off.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: keys, telemetry and tracing off, judge model, inputs
# ============================================================================
import os
import re
import json
import warnings
from collections import Counter, defaultdict

from dotenv import load_dotenv

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"   # no DeepEval usage telemetry
os.environ["RAGAS_DO_NOT_TRACK"] = "true"         # no RAGAS usage telemetry
warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm's note about Jupyter widgets
load_dotenv(os.path.join("..", "..", ".env"))
os.environ["LANGSMITH_TRACING"] = "false"         # nothing to trace: the agent doesn't run here

JUDGE_MODEL = "gpt-4o"
EMBEDDING_MODEL = "text-embedding-3-small"
DECLINE = "I'm sorry, I don't have information about that. I can connect you with a human agent."

if not os.path.exists(os.path.join("data", "golden_runs.json")):
    raise RuntimeError("No data/golden_runs.json. Run notebook 04 first: it records the baseline run.")
with open(os.path.join("data", "golden_set.json"), encoding="utf-8") as f:
    golden_set = json.load(f)
with open(os.path.join("data", "golden_runs.json"), encoding="utf-8") as f:
    golden_runs = json.load(f)
with open(os.path.join("data", "kb_chunks.json"), encoding="utf-8") as f:
    chunk_text = {chunk["chunk_id"]: chunk["text"] for chunk in json.load(f)}

print("✅ Setup complete; judge model:", JUDGE_MODEL)
print(f"📂 {len(golden_set)} golden cases, {len(golden_runs)} recorded runs")

---
## 🧭 Part 1: Which answers to grade

Answer quality only makes sense for answers the RAG agent wrote from retrieved chunks. Each
knowledge case (`simple`, `multi_hop`, `out_of_scope`) falls into one of three groups:

- **Answerable, reached the RAG agent**: graded with every metric in this notebook
- **Out of scope, reached the RAG agent**: graded on one thing, whether it declined
- **Sent somewhere else by the supervisor**: no chunks to grade against; a routing failure,
  scored in notebook 07 and recorded here as such

In [ ]:
# ============================================================================
# SPLIT THE KNOWLEDGE CASES INTO THE THREE GROUPS
# ============================================================================
cases_by_id = {case["id"]: case for case in golden_set}
answerable_runs, out_of_scope_runs, misrouted_runs = [], [], []

for run in golden_runs:
    if run["slice"] not in ("simple", "multi_hop", "out_of_scope"):
        continue
    if run["route"] != "rag_agent":
        misrouted_runs.append(run)
    elif run["slice"] == "out_of_scope":
        out_of_scope_runs.append(run)
    else:
        answerable_runs.append(run)

print(f"✍️ answerable, graded in full: {len(answerable_runs)}", dict(Counter(r["slice"] for r in answerable_runs)))
print(f"🚫 out of scope, decline check: {len(out_of_scope_runs)}")
print(f"🧭 sent elsewhere by the supervisor: {len(misrouted_runs)}")
for run in misrouted_runs:
    print(f"   {run['case_id']:<16} → {run['route']:<14} {run['question'][:60]}")

---
## 🚫 Part 2: Did it decline exactly when it should?

The RAG prompt tells the model to reply with one exact sentence when the articles don't hold
the answer. Because the sentence is fixed, checking it needs no judge: a string comparison is
exact and free. Two errors matter, in opposite directions:
- **Missed decline**: an out-of-scope question got an invented answer (a hallucination)
- **False decline**: an answerable question got "I don't know" (unhelpful, but safe)

In [ ]:
# ============================================================================
# DECLINE CORRECTNESS: a fixed sentence, so a string check is enough
# ============================================================================
def is_decline(answer):
    return DECLINE.lower() in answer.lower()

correct_declines = [run for run in out_of_scope_runs if is_decline(run["answer"])]
false_declines = [run for run in answerable_runs if is_decline(run["answer"])]

print(f"🧮 out of scope declined: {len(correct_declines)} of {len(out_of_scope_runs)} "
      f"({len(correct_declines) / len(out_of_scope_runs):.0%})")
for run in out_of_scope_runs:
    if not is_decline(run["answer"]):
        print(f"   ❌ missed decline {run['case_id']}: {run['answer'][:120]}")
print(f"🧮 answerable questions declined: {len(false_declines)} of {len(answerable_runs)}")
for run in false_declines:
    print(f"   ⚠️ false decline {run['case_id']}: {run['question']}")
print("Reading: a missed decline is the worse error; it means the agent invents policy.")

---
## 🧾 Part 3: Reference-free metrics

These need only the question, the chunks and the answer, so they also work on live traffic
where nobody has written a reference (notebook 14).

| Metric | Library | Question | Higher is |
|---|---|---|---|
| Faithfulness | RAGAS, DeepEval | Is every claim in the answer supported by the retrieved chunks? | Better |
| Answer relevancy | RAGAS, DeepEval | Does the answer address the question that was asked? | Better |
| Hallucination | DeepEval | Does the answer agree with the *correct* chunks (the labelled ones)? | Better |

> **Note:** in DeepEval 4.2.3, `HallucinationMetric` scores the share of the given contexts the
> answer agrees with, so **higher is better** and a case passes when the score is at or above
> the threshold. Older DeepEval documentation describes the opposite direction; check the
> installed version before reading the number.

In [ ]:
# ============================================================================
# RAGAS: FAITHFULNESS AND ANSWER RELEVANCY (one case at a time)
# ============================================================================
# 💰 About 4 gpt-4o calls per case. RAGAS scores with async calls: it needs the async client.
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
from ragas.metrics.collections import Faithfulness, AnswerRelevancy

async_client = AsyncOpenAI()
ragas_judge = llm_factory(JUDGE_MODEL, client=async_client)
ragas_embeddings = RagasOpenAIEmbeddings(client=async_client, model=EMBEDDING_MODEL)
ragas_faithfulness = Faithfulness(llm=ragas_judge)
ragas_relevancy = AnswerRelevancy(llm=ragas_judge, embeddings=ragas_embeddings)

scores = defaultdict(dict)        # case id → every score this notebook computes
for run in answerable_runs:
    faithfulness = await ragas_faithfulness.ascore(user_input=run["question"], response=run["answer"],
                                                   retrieved_contexts=run["contexts"])
    relevancy = await ragas_relevancy.ascore(user_input=run["question"], response=run["answer"])
    scores[run["case_id"]]["ragas_faithfulness"] = faithfulness.value
    scores[run["case_id"]]["ragas_answer_relevancy"] = relevancy.value
print(f"✅ RAGAS scored {len(answerable_runs)} answers")

In [ ]:
# ============================================================================
# DEEPEVAL: FAITHFULNESS, ANSWER RELEVANCY AND HALLUCINATION
# ============================================================================
# 💰 About 6 gpt-4o calls per case. _show_indicator=False stops a progress bar per call.
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, HallucinationMetric
from deepeval.test_case import LLMTestCase

deepeval_faithfulness = FaithfulnessMetric(model=JUDGE_MODEL, include_reason=False)
deepeval_relevancy = AnswerRelevancyMetric(model=JUDGE_MODEL, include_reason=False)
deepeval_hallucination = HallucinationMetric(model=JUDGE_MODEL, include_reason=False)

for run in answerable_runs:
    case = cases_by_id[run["case_id"]]
    correct_chunks = [chunk_text[chunk_id] for chunk_id in case["relevant_chunk_ids"]]
    test_case = LLMTestCase(input=run["question"], actual_output=run["answer"],
                            retrieval_context=run["contexts"],   # what the model was given
                            context=correct_chunks)              # what is known to be correct
    scores[run["case_id"]]["deepeval_faithfulness"] = await deepeval_faithfulness.a_measure(test_case, _show_indicator=False)
    scores[run["case_id"]]["deepeval_answer_relevancy"] = await deepeval_relevancy.a_measure(test_case, _show_indicator=False)
    scores[run["case_id"]]["deepeval_hallucination"] = await deepeval_hallucination.a_measure(test_case, _show_indicator=False)
print(f"✅ DeepEval scored {len(answerable_runs)} answers")

In [ ]:
# ============================================================================
# REFERENCE-FREE SCORES, PER SLICE
# ============================================================================
def mean_of(metric_name, case_ids):
    values = [scores[case_id][metric_name] for case_id in case_ids if scores[case_id][metric_name] is not None]
    return sum(values) / len(values)

REFERENCE_FREE = ["ragas_faithfulness", "deepeval_faithfulness", "ragas_answer_relevancy",
                  "deepeval_answer_relevancy", "deepeval_hallucination"]
for slice_name in ["simple", "multi_hop"]:
    case_ids = [run["case_id"] for run in answerable_runs if run["slice"] == slice_name]
    print(f"[{slice_name}] {len(case_ids)} answers")
    for metric_name in REFERENCE_FREE:
        print(f"   {metric_name:<26} {mean_of(metric_name, case_ids):.3f}")
print("Reading: faithfulness ≥ 0.90 is the target (spec §4). RAGAS answer relevancy is lower than")
print("DeepEval's because it rebuilds questions from the answer and compares them by embedding;")
print("short answers full of [chunk-id] citations rebuild poorly even when they are right.")

---
## 📐 Part 4: Reference-based metrics

Now compare each answer with the golden set's `reference_answer`, from the bluntest check to
the most forgiving:

| Metric | How | Good for |
|---|---|---|
| Exact match | The normalised answer equals the normalised reference | IDs, dates, yes/no |
| Token F1 | Overlap of words between answer and reference | Short factual answers |
| Factual correctness (RAGAS) | Split both into claims; precision and recall of matching claims | Longer answers |
| Semantic similarity (RAGAS) | Cosine similarity of the two embeddings | A cheap "same meaning?" signal |
| Correctness (DeepEval `GEval`) | A judge follows written steps and scores 0–1 | When wording varies a lot |

Citations like `[shipping-01]` are stripped first: they are part of the answer format, not
of what the answer says.

In [ ]:
# ============================================================================
# EXACT MATCH AND TOKEN F1 (stdlib, free)
# ============================================================================
def normalise(text):
    text = re.sub(r"\[[a-z0-9-]+\]", " ", text.lower())      # drop [chunk-id] citations
    return re.findall(r"[a-z0-9$.]+", text)                     # keep words, numbers and prices

def token_f1(answer, reference):
    answer_words, reference_words = normalise(answer), normalise(reference)
    common = sum((Counter(answer_words) & Counter(reference_words)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(answer_words), common / len(reference_words)
    return 2 * precision * recall / (precision + recall)

for run in answerable_runs:
    reference = cases_by_id[run["case_id"]]["reference_answer"]
    scores[run["case_id"]]["exact_match"] = 1.0 if normalise(run["answer"]) == normalise(reference) else 0.0
    scores[run["case_id"]]["token_f1"] = token_f1(run["answer"], reference)

all_ids = [run["case_id"] for run in answerable_runs]
print(f"🧮 exact match {mean_of('exact_match', all_ids):.3f}   token F1 {mean_of('token_f1', all_ids):.3f}")
print("Reading: near zero, and that is the lesson: a correct sentence rarely matches a reference")
print("word for word. These two only suit answers that are a single ID, date or number.")

In [ ]:
# ============================================================================
# RAGAS: FACTUAL CORRECTNESS AND SEMANTIC SIMILARITY
# ============================================================================
# 💰 About 3 gpt-4o calls per case for factual correctness; similarity is two embeddings.
from ragas.metrics.collections import FactualCorrectness, SemanticSimilarity

ragas_factual = FactualCorrectness(llm=ragas_judge)              # claim-level F1 by default
ragas_similarity = SemanticSimilarity(embeddings=ragas_embeddings)

for run in answerable_runs:
    reference = cases_by_id[run["case_id"]]["reference_answer"]
    factual = await ragas_factual.ascore(response=run["answer"], reference=reference)
    similarity = await ragas_similarity.ascore(reference=reference, response=run["answer"])
    scores[run["case_id"]]["ragas_factual_correctness"] = factual.value
    scores[run["case_id"]]["ragas_semantic_similarity"] = similarity.value
print(f"✅ RAGAS scored {len(answerable_runs)} answers against their references")

In [ ]:
# ============================================================================
# DEEPEVAL GEval: A CORRECTNESS JUDGE WITH WRITTEN STEPS
# ============================================================================
# 💰 One or two gpt-4o calls per case. The steps are the rubric; writing them is the real work.
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

correctness_judge = GEval(
    name="Correctness",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    evaluation_steps=[
        "List the facts in the expected output that answer the input.",
        "Check whether the actual output states each of those facts; wording may differ.",
        "Penalise heavily any fact in the actual output that contradicts the expected output.",
        "Ignore citations in square brackets and extra detail that does not contradict anything.",
    ],
    model=JUDGE_MODEL,
)
for run in answerable_runs:
    test_case = LLMTestCase(input=run["question"], actual_output=run["answer"],
                            expected_output=cases_by_id[run["case_id"]]["reference_answer"])
    scores[run["case_id"]]["geval_correctness"] = await correctness_judge.a_measure(test_case, _show_indicator=False)
print(f"✅ GEval scored {len(answerable_runs)} answers")

In [ ]:
# ============================================================================
# REFERENCE-BASED SCORES, PER SLICE
# ============================================================================
REFERENCE_BASED = ["exact_match", "token_f1", "ragas_factual_correctness",
                   "ragas_semantic_similarity", "geval_correctness"]
for slice_name in ["simple", "multi_hop"]:
    case_ids = [run["case_id"] for run in answerable_runs if run["slice"] == slice_name]
    print(f"[{slice_name}] {len(case_ids)} answers")
    for metric_name in REFERENCE_BASED:
        print(f"   {metric_name:<26} {mean_of(metric_name, case_ids):.3f}")
print("Reading: the metrics rank the same answers very differently. Semantic similarity is high for")
print("anything on-topic, so it can't tell right from nearly right; the claim-level and GEval judges can.")

In [ ]:
# ============================================================================
# THE LOWEST-SCORING ANSWERS ON THE CORRECTNESS JUDGE
# ============================================================================
lowest = sorted(answerable_runs, key=lambda run: scores[run["case_id"]]["geval_correctness"])[:4]
for run in lowest:
    case_scores = scores[run["case_id"]]
    print(f"[{run['case_id']}] GEval {case_scores['geval_correctness']:.2f}   "
          f"RAGAS factual {case_scores['ragas_factual_correctness']:.2f}   "
          f"faithfulness {case_scores['ragas_faithfulness']:.2f}")
    print(f"   ❓ {run['question']}")
    print(f"   🤖 {run['answer'][:220]}")
    print(f"   📝 {cases_by_id[run['case_id']]['reference_answer'][:220]}")

---
## 🧑‍⚖️ Part 5: Who is to blame?

A wrong answer has one of three causes, and each needs a different fix. Combining the scores
above tells them apart, with no new model calls (spec §6):

| Retrieval found every labelled chunk? | Answer faithful to its chunks? | Answer correct? | Verdict | Fix |
|---|---|---|---|---|
| — | — | Yes | ok | — |
| No | — | No | Retrieval failure | Retriever, chunking, k |
| Yes | No | No | Generation failure | Prompt, model |
| Yes | Yes | No | Test case or KB wrong | The golden set or the articles |

The thresholds: retrieval found every labelled chunk in the top 4 (label recall 1.0); faithful
means RAGAS faithfulness ≥ 0.90; correct means `GEval` correctness ≥ 0.5.

In [ ]:
# ============================================================================
# A VERDICT FOR EVERY ANSWER
# ============================================================================
def label_recall(run):
    relevant = cases_by_id[run["case_id"]]["relevant_chunk_ids"]
    return len(set(run["retrieved_chunk_ids"]) & set(relevant)) / len(relevant)

def blame(run):
    case_scores = scores[run["case_id"]]
    if case_scores["geval_correctness"] >= 0.5:
        return "ok"
    if case_scores["label_recall"] < 1.0:
        return "retrieval failure"
    if not (case_scores["ragas_faithfulness"] is not None and case_scores["ragas_faithfulness"] >= 0.90):
        return "generation failure"
    return "test case or KB wrong"

for run in answerable_runs:
    scores[run["case_id"]]["label_recall"] = label_recall(run)
    scores[run["case_id"]]["verdict"] = blame(run)
print(Counter(scores[run["case_id"]]["verdict"] for run in answerable_runs))

In [ ]:
# ============================================================================
# THE BLAME TABLE, BY SLICE
# ============================================================================
VERDICTS = ["ok", "retrieval failure", "generation failure", "test case or KB wrong"]
table = defaultdict(Counter)
for run in answerable_runs:
    table[run["slice"]][scores[run["case_id"]]["verdict"]] += 1
for run in misrouted_runs:
    table[run["slice"]]["routing failure"] += 1

print(f"{'slice':<10} " + " ".join(f"{verdict:>22}" for verdict in VERDICTS + ["routing failure"]))
for slice_name in ["simple", "multi_hop"]:
    print(f"{slice_name:<10} " + " ".join(f"{table[slice_name][verdict]:>22}" for verdict in VERDICTS + ["routing failure"]))
print()
for run in answerable_runs:
    if scores[run["case_id"]]["verdict"] != "ok":
        print(f"   {scores[run['case_id']]['verdict']:<22} {run['case_id']}: {run['question'][:70]}")
print("Reading: with retrieval near perfect (notebook 04), the failures that remain are the")
print("generator's or the test set's own, plus the questions the supervisor never sent to RAG.")

---
## 💾 Part 6: Save the scores

Notebook 12 compares slices and runs statistics over these per-case scores, so they are saved
with the `run_id` that links each one to its LangSmith trace.

In [ ]:
# ============================================================================
# SAVE PER-CASE RAG SCORES → data/rag_scores.json
# ============================================================================
rag_scores = {}
for run in answerable_runs:
    rag_scores[run["case_id"]] = {"run_id": run["run_id"], "slice": run["slice"], "route": run["route"],
                                  "declined": is_decline(run["answer"]), **scores[run["case_id"]]}
for run in out_of_scope_runs:
    rag_scores[run["case_id"]] = {"run_id": run["run_id"], "slice": run["slice"], "route": run["route"],
                                  "declined": is_decline(run["answer"]),
                                  "verdict": "ok" if is_decline(run["answer"]) else "missed decline"}
for run in misrouted_runs:
    rag_scores[run["case_id"]] = {"run_id": run["run_id"], "slice": run["slice"], "route": run["route"],
                                  "verdict": "routing failure"}

with open(os.path.join("data", "rag_scores.json"), "w", encoding="utf-8") as f:
    json.dump(rag_scores, f, indent=2, ensure_ascii=False)
print(f"✅ Saved scores for {len(rag_scores)} knowledge cases → data/rag_scores.json")

---
## 📝 Summary

In this notebook, we learned:

### 1. Declines
- **A fixed sentence is checked with a string compare**: free, exact, no judge needed
- **Two directions**: a missed decline invents policy; a false decline is unhelpful but safe

### 2. Reference-free metrics
- **Faithfulness and relevancy** need no reference, so they also run in production
- **Same name, different behaviour**: RAGAS and DeepEval relevancy disagree on short cited answers;
  DeepEval 4.2.3's hallucination score is higher-is-better

### 3. Reference-based metrics
- **Exact match and F1 fail on sentences**; claim-level and rubric judges handle rewording

### 4. Blame
- **Recall + faithfulness + correctness** split each failure between retriever, generator and test case

### Next Steps
- Notebook 06 turns to the account agent: did it call the right tool with the right arguments?